# Validation and cross-round checks for ESS Round 4 and ESS Round 8

This notebook validates the four curated respondent-level datasets produced by the two curation notebooks.

It performs four distinct checks:

1. verifies the dimensions, schemas, missingness rules, identifiers, and official weight columns of all four outputs;
2. independently reconstructs the ESS8 descriptive-statistics validation against Supplementary Table A2;
3. compares the ESS4 Python output directly with Jochem van Noord's saved deterministic `df_ESS4.RData` object;
4. produces a descriptive comparison of the 12 belief concepts shared by ESS4 and ESS8.

The ESS4 comparison is respondent-level and belief-level. The two rounds are **not** compared respondent by respondent because they contain different survey respondents and different country coverage.

This notebook does not run CCA and does not modify any raw ESS file or reference file.

## Before running the notebook

The Python package `pyreadr` is required to open Van Noord's `df_ESS4.RData`.

Install it once in the same Python environment used by this Jupyter kernel:

```python
%pip install pyreadr
```

After installation, restart the kernel and run all cells.

The notebook expects the project structure already agreed for this repository, including:

```text
data/processed/
reference/van_noord/ESS Round 4/data/df_ESS4.RData
src/ess4_config.py
src/ess8_config.py
src/ess_curation_common.py
```

## Step 1 — Locate the project, import the shared configuration, and define paths

The notebook may be launched from either the project root or the `notebooks/` folder. The code below locates the root folder automatically and defines every required input and output path.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def locate_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if all((candidate / folder).is_dir() for folder in ("data", "notebooks", "src")):
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root containing data/, notebooks/, and src/."
    )


PROJECT_ROOT = locate_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import ess4_config as ess4
from src import ess8_config as ess8
from src.ess_curation_common import (
    summarise_beliefs,
    validate_weight_split,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

ESS4_WITHOUT_PATH = PROCESSED_DIR / ess4.OUTPUT_FILENAMES["without_weights"]
ESS4_WITH_PATH = PROCESSED_DIR / ess4.OUTPUT_FILENAMES["with_weights"]
ESS8_WITHOUT_PATH = PROCESSED_DIR / ess8.OUTPUT_FILENAMES["without_weights"]
ESS8_WITH_PATH = PROCESSED_DIR / ess8.OUTPUT_FILENAMES["with_weights"]

ESS4_REFERENCE_RDATA_PATH = PROJECT_ROOT.joinpath(*ess4.REFERENCE_RDATA_PARTS)

ESS4_VALIDATION_PATH = PROCESSED_DIR / ess4.OUTPUT_FILENAMES["validation"]
ESS8_VALIDATION_PATH = PROCESSED_DIR / ess8.OUTPUT_FILENAMES["validation"]
CROSS_ROUND_DESCRIPTIVES_PATH = (
    PROCESSED_DIR / "ess4_ess8_shared_belief_descriptives.csv"
)

required_paths = (
    ESS4_WITHOUT_PATH,
    ESS4_WITH_PATH,
    ESS8_WITHOUT_PATH,
    ESS8_WITH_PATH,
    ESS4_REFERENCE_RDATA_PATH,
)

missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "The following required files were not found:\n"
        + "\n".join(f"- {path}" for path in missing_paths)
    )

print("Project root:", PROJECT_ROOT)
print("All required curated datasets and the ESS4 reference RData file were found.")

Project root: /Users/karan/Desktop/SSM-MERC/polarization/data_curation
All required curated datasets and the ESS4 reference RData file were found.


## Step 2 — Load the four curated CSV files

The weighted and unweighted versions for a given round must contain the same respondents and belief values. The only permitted difference is the presence of the four official ESS weight columns.

In [2]:
ess4_without = pd.read_csv(ESS4_WITHOUT_PATH, low_memory=False)
ess4_with = pd.read_csv(ESS4_WITH_PATH, low_memory=False)
ess8_without = pd.read_csv(ESS8_WITHOUT_PATH, low_memory=False)
ess8_with = pd.read_csv(ESS8_WITH_PATH, low_memory=False)

loaded_summary = pd.DataFrame(
    [
        {
            "dataset": "ESS4 without weights",
            "rows": len(ess4_without),
            "columns": ess4_without.shape[1],
        },
        {
            "dataset": "ESS4 with weights",
            "rows": len(ess4_with),
            "columns": ess4_with.shape[1],
        },
        {
            "dataset": "ESS8 without weights",
            "rows": len(ess8_without),
            "columns": ess8_without.shape[1],
        },
        {
            "dataset": "ESS8 with weights",
            "rows": len(ess8_with),
            "columns": ess8_with.shape[1],
        },
    ]
)

display(loaded_summary)

,dataset,rows,columns
0,ESS4 without weights,45268,34
1,ESS4 with weights,45268,38
2,ESS8 without weights,37118,36
3,ESS8 with weights,37118,40


## Step 3 — Validate schemas, sample rules, identifiers, value ranges, and weight separation

For each round, this step checks that:

- the expected number of respondents and countries is present;
- the output columns are in the configured order;
- `ess_unique_id` is complete and unique;
- every retained respondent has at most two missing constructed beliefs;
- all observed belief values lie between 0 and 1;
- all four official ESS weights are present and complete in the with-weights file;
- removing the four weights from the with-weights file reproduces the without-weights file exactly.

In [3]:
def validate_round_outputs(
    *,
    round_label: str,
    without_weights: pd.DataFrame,
    with_weights: pd.DataFrame,
    config,
) -> dict:
    assert without_weights.shape == config.EXPECTED_WITHOUT_WEIGHTS_SHAPE
    assert with_weights.shape == config.EXPECTED_WITH_WEIGHTS_SHAPE

    assert list(without_weights.columns) == list(config.OUTPUT_COLUMNS_WITHOUT_WEIGHTS)
    assert list(with_weights.columns) == list(config.OUTPUT_COLUMNS_WITH_WEIGHTS)

    assert len(without_weights) == config.EXPECTED_FINAL_N
    assert without_weights["cntry"].nunique(dropna=True) == config.EXPECTED_FINAL_COUNTRIES

    assert without_weights["ess_unique_id"].notna().all()
    assert without_weights["ess_unique_id"].is_unique
    assert without_weights["idno"].notna().all()
    assert without_weights["cntry"].notna().all()

    assert without_weights["n_belief_missing"].le(
        config.MAXIMUM_MISSING_BELIEFS
    ).all()
    assert (
        without_weights["n_belief_available"]
        == len(config.BELIEF_COLUMNS) - without_weights["n_belief_missing"]
    ).all()

    belief_values = without_weights.loc[:, config.BELIEF_COLUMNS]
    observed_minimum = belief_values.min(skipna=True).min()
    observed_maximum = belief_values.max(skipna=True).max()
    assert observed_minimum >= 0.0
    assert observed_maximum <= 1.0

    validate_weight_split(
        without_weights,
        with_weights,
        config.WEIGHT_COLUMNS,
        require_complete_weights=True,
    )

    return {
        "round": round_label,
        "respondents": len(without_weights),
        "countries": without_weights["cntry"].nunique(dropna=True),
        "beliefs": len(config.BELIEF_COLUMNS),
        "minimum_observed_belief": observed_minimum,
        "maximum_observed_belief": observed_maximum,
        "maximum_missing_beliefs": int(
            without_weights["n_belief_missing"].max()
        ),
        "weights_complete": bool(
            with_weights.loc[:, config.WEIGHT_COLUMNS].notna().all().all()
        ),
        "weighted_unweighted_match": True,
    }


round_checks = pd.DataFrame(
    [
        validate_round_outputs(
            round_label=ess4.ROUND_LABEL,
            without_weights=ess4_without,
            with_weights=ess4_with,
            config=ess4,
        ),
        validate_round_outputs(
            round_label=ess8.ROUND_LABEL,
            without_weights=ess8_without,
            with_weights=ess8_with,
            config=ess8,
        ),
    ]
)

display(round_checks)

,round,respondents,countries,beliefs,minimum_observed_belief,maximum_observed_belief,maximum_missing_beliefs,weights_complete,weighted_unweighted_match
0,ESS Round 4,45268,29,19,0.0,1.0,2,True,True
1,ESS Round 8,37118,23,20,0.0,1.0,2,True,True


## Step 4 — Reconstruct the ESS8 validation against Supplementary Table A2

This step independently calculates the non-missing count, mean, and sample standard deviation of every ESS8 belief variable. It then compares these values with the published targets stored in `src/ess8_config.py`.

The published means and standard deviations are reported to two decimal places, so those two statistics are validated after rounding to two decimal places. The non-missing counts must match exactly.

In [4]:
ess8_reproduced = summarise_beliefs(
    ess8_without,
    ess8.BELIEF_COLUMNS,
)

ess8_paper = pd.DataFrame(
    ess8.PAPER_STATISTICS,
    columns=ess8.PAPER_STATISTICS_COLUMNS,
)

ess8_validation = ess8_paper.merge(
    ess8_reproduced,
    on="belief_variable",
    how="left",
    validate="one_to_one",
)

ess8_validation["mean_reproduced_round2"] = (
    ess8_validation["mean_reproduced"].round(2)
)
ess8_validation["sd_reproduced_round2"] = (
    ess8_validation["sd_reproduced"].round(2)
)
ess8_validation["N_matches"] = (
    ess8_validation["N_paper"] == ess8_validation["N_reproduced"]
)
ess8_validation["mean_matches_round2"] = (
    ess8_validation["mean_paper"]
    == ess8_validation["mean_reproduced_round2"]
)
ess8_validation["sd_matches_round2"] = (
    ess8_validation["sd_paper"]
    == ess8_validation["sd_reproduced_round2"]
)

assert ess8_validation[
    ["N_matches", "mean_matches_round2", "sd_matches_round2"]
].all().all()

ess8_validation.to_csv(ESS8_VALIDATION_PATH, index=False)

print("All 20 ESS8 belief variables match Supplementary Table A2.")
print("Validation file written to:", ESS8_VALIDATION_PATH)
display(ess8_validation)

All 20 ESS8 belief variables match Supplementary Table A2.
Validation file written to: /Users/karan/Desktop/SSM-MERC/polarization/data_curation/data/processed/ess8_belief_validation_against_paper.csv


,belief_variable,N_paper,mean_paper,sd_paper,N_reproduced,mean_reproduced,sd_reproduced,mean_reproduced_round2,sd_reproduced_round2,N_matches,mean_matches_round2,sd_matches_round2
0,left_right_identification,34248,0.51,0.22,34248,0.512920,0.223370,0.51,0.22,True,True,True
1,gender_inequality,37038,0.23,0.27,37038,0.227955,0.269111,0.23,0.27,True,True,True
2,anti_lgbt,36098,0.34,0.27,36098,0.336637,0.270381,0.34,0.27,True,True,True
3,euroscepticism,35848,0.51,0.27,35848,0.508968,0.266828,0.51,0.27,True,True,True
4,anti_immigration,36459,0.45,0.27,36459,0.451734,0.265234,0.45,0.27,True,True,True
5,anti_egalitarianism,36772,0.38,0.19,36772,0.380239,0.193550,0.38,0.19,True,True,True
6,benefits_harm_economy,35956,0.49,0.23,35956,0.490534,0.226702,0.49,0.23,True,True,True
7,benefits_harm_society,36801,0.41,0.22,36801,0.410132,0.218079,0.41,0.22,True,True,True
8,welfare_chauvinism,36441,0.54,0.26,36441,0.543090,0.258760,0.54,0.26,True,True,True
9,anti_economic_interventionism,36927,0.24,0.16,36927,0.244343,0.159874,0.24,0.16,True,True,True


## Step 5 — Load Van Noord's deterministic ESS4 reference object

`df_ESS4.RData` is the saved output of the supplied `Data cleaning_ESS4.R` script. Unlike the later modified CCA results, this cleaning-stage object is deterministic and can be used for exact validation.

The code accepts either an R object named `df` or a file containing a single data-frame object.

In [5]:
try:
    import pyreadr
except ImportError as exc:
    raise ImportError(
        "pyreadr is required for this notebook. Run `%pip install pyreadr`, "
        "restart the kernel, and run the notebook again."
    ) from exc

r_objects = pyreadr.read_r(str(ESS4_REFERENCE_RDATA_PATH))

if "df" in r_objects:
    ess4_reference_raw = r_objects["df"]
elif len(r_objects) == 1:
    ess4_reference_raw = next(iter(r_objects.values()))
else:
    raise KeyError(
        "Could not identify the ESS4 reference dataframe inside the RData file. "
        f"Objects found: {list(r_objects)}"
    )

assert isinstance(ess4_reference_raw, pd.DataFrame)
assert len(ess4_reference_raw) == ess4.EXPECTED_FINAL_N

print("Objects found in RData:", list(r_objects))
print("Reference rows:", len(ess4_reference_raw))
print("Reference columns:", ess4_reference_raw.shape[1])
display(ess4_reference_raw.head())

Objects found in RData: ['df']
Reference rows: 45268
Reference columns: 29


,id,essid,country,education,hhincome,female,age,religious,urbanization,ethnic_minority,...,anti_interventionism,harsh_sentences,anti_mil_democracy,science_environment,government_spending,regressive_taxes,regressive_benefits,age_prejudice,authoritarianism,anti_libertarianism
0,1,10202.0,BE,Higher educated,4.0,Male,36.0,Non-religious,5.0,Ethnic majority,...,0.316667,0.25,0.75,0.25,0.5,0.5,0.0,0.6,0.48,0.40
1,2,10203.0,BE,Higher educated,7.0,Female,26.0,Non-religious,5.0,Ethnic majority,...,0.300000,0.75,0.25,0.50,0.5,0.5,0.0,0.3,0.60,0.44
2,3,10207.0,BE,Higher educated,10.0,Male,69.0,Religious,5.0,Ethnic majority,...,0.550000,0.50,0.75,0.75,0.5,0.0,0.0,0.6,0.56,0.20
3,4,10208.0,BE,Higher educated,7.0,Female,77.0,Religious,5.0,Ethnic majority,...,0.500000,0.50,0.25,0.25,0.5,0.0,0.0,0.2,0.68,0.44
4,5,10302.0,BE,Middle educated,7.0,Male,27.0,Non-religious,5.0,Ethnic majority,...,0.450000,0.25,0.75,0.25,0.5,0.5,0.5,0.0,0.64,0.36


## Step 6 — Harmonize the ESS4 reference column names and factor labels

Van Noord's R object uses different names from the transparent ESS-style names in the Python output. It also stores several demographics as labelled factors.

This step maps the R object to the Python conventions without altering the underlying responses:

- `essid` → `idno`;
- `country` → `cntry`;
- R factor labels for education, gender, religion, and ethnic-minority status are mapped back to the corresponding numeric categories used in the curated Python file;
- all 19 R belief columns are renamed to their Python output names.

In [6]:
R_TO_PYTHON_BELIEF = {
    "lrscale": "left_right_identification",
    "gender_inequality": "gender_inequality",
    "anti_lgbt": "anti_lgbt",
    "euroscepticism": "euroscepticism",
    "anti_immigration": "anti_immigration",
    "anti_egalitarianism": "anti_egalitarianism",
    "benefits_eco": "benefits_harm_economy",
    "benefits_soc": "benefits_harm_society",
    "welfare_chauvinism": "welfare_chauvinism",
    "anti_interventionism": "anti_economic_interventionism",
    "harsh_sentences": "harsh_sentences",
    "anti_mil_democracy": "anti_militant_democracy",
    "science_environment": "no_science_environment_solution",
    "government_spending": "anti_government_spending",
    "regressive_taxes": "regressive_taxes",
    "regressive_benefits": "regressive_benefits",
    "age_prejudice": "age_prejudice",
    "authoritarianism": "authoritarianism",
    "anti_libertarianism": "anti_libertarianism",
}

required_reference_columns = {
    "essid",
    "country",
    "education",
    "hhincome",
    "female",
    "age",
    "religious",
    "urbanization",
    "ethnic_minority",
    *R_TO_PYTHON_BELIEF.keys(),
}

missing_reference_columns = sorted(
    required_reference_columns - set(ess4_reference_raw.columns)
)
if missing_reference_columns:
    raise KeyError(
        "Required columns are missing from df_ESS4.RData: "
        f"{missing_reference_columns}"
    )


def map_factor_to_numeric(
    series: pd.Series,
    *,
    label_mapping: dict[str, float],
    numeric_code_mapping: dict[float, float],
) -> pd.Series:
    text_values = series.astype("string").str.strip()
    from_labels = text_values.map(label_mapping)

    numeric_values = pd.to_numeric(series, errors="coerce")
    from_codes = numeric_values.map(numeric_code_mapping)

    return pd.to_numeric(from_labels.fillna(from_codes), errors="coerce")


country_text = ess4_reference_raw["country"].astype("string").str.strip()
country_name_to_code = {name: code for code, name in ess4.COUNTRY_LABELS.items()}
country_aliases = {
    "Great Britain": "GB",
    "United Kingdom": "GB",
    "Czech Republic": "CZ",
    "Czechia": "CZ",
    "Russia": "RU",
    "Russian Federation": "RU",
    "Turkey": "TR",
    "Türkiye": "TR",
}
country_name_to_code.update(country_aliases)

ess4_reference = pd.DataFrame(index=ess4_reference_raw.index)
ess4_reference["idno"] = pd.to_numeric(
    ess4_reference_raw["essid"],
    errors="coerce",
).astype("Int64")
ess4_reference["cntry"] = country_text.where(
    country_text.isin(ess4.COUNTRY_LABELS),
    country_text.map(country_name_to_code),
)

ess4_reference["education_3cat"] = map_factor_to_numeric(
    ess4_reference_raw["education"],
    label_mapping={
        "Lower educated": 1,
        "Middle educated": 2,
        "Higher educated": 3,
    },
    numeric_code_mapping={1: 1, 2: 2, 3: 3},
)
ess4_reference["hinctnta"] = pd.to_numeric(
    ess4_reference_raw["hhincome"],
    errors="coerce",
)
ess4_reference["gndr"] = map_factor_to_numeric(
    ess4_reference_raw["female"],
    label_mapping={"Male": 1, "Female": 2},
    numeric_code_mapping={1: 1, 2: 2},
)
ess4_reference["agea"] = pd.to_numeric(
    ess4_reference_raw["age"],
    errors="coerce",
)
ess4_reference["rlgblg"] = map_factor_to_numeric(
    ess4_reference_raw["religious"],
    label_mapping={"Non-religious": 2, "Religious": 1},
    numeric_code_mapping={1: 2, 2: 1},
)
ess4_reference["urbanization"] = pd.to_numeric(
    ess4_reference_raw["urbanization"],
    errors="coerce",
)
ess4_reference["blgetmg"] = map_factor_to_numeric(
    ess4_reference_raw["ethnic_minority"],
    label_mapping={"Ethnic majority": 2, "Ethnic minority": 1},
    numeric_code_mapping={1: 2, 2: 1},
)

for r_column, python_column in R_TO_PYTHON_BELIEF.items():
    ess4_reference[python_column] = pd.to_numeric(
        ess4_reference_raw[r_column],
        errors="coerce",
    )

assert ess4_reference["idno"].notna().all()
assert ess4_reference["cntry"].notna().all()
assert ess4_reference[["cntry", "idno"]].duplicated().sum() == 0

print("Harmonized ESS4 reference dimensions:", ess4_reference.shape)
display(ess4_reference.head())

Harmonized ESS4 reference dimensions: (45268, 28)


,idno,cntry,education_3cat,hinctnta,gndr,agea,rlgblg,urbanization,blgetmg,left_right_identification,...,anti_economic_interventionism,harsh_sentences,anti_militant_democracy,no_science_environment_solution,anti_government_spending,regressive_taxes,regressive_benefits,age_prejudice,authoritarianism,anti_libertarianism
0,10202,BE,3.0,4.0,1.0,36.0,2.0,5.0,2.0,0.7,...,0.316667,0.25,0.75,0.25,0.5,0.5,0.0,0.6,0.48,0.40
1,10203,BE,3.0,7.0,2.0,26.0,2.0,5.0,2.0,0.6,...,0.300000,0.75,0.25,0.50,0.5,0.5,0.0,0.3,0.60,0.44
2,10207,BE,3.0,10.0,1.0,69.0,1.0,5.0,2.0,0.8,...,0.550000,0.50,0.75,0.75,0.5,0.0,0.0,0.6,0.56,0.20
3,10208,BE,3.0,7.0,2.0,77.0,1.0,5.0,2.0,0.6,...,0.500000,0.50,0.25,0.25,0.5,0.0,0.0,0.2,0.68,0.44
4,10302,BE,2.0,7.0,1.0,27.0,2.0,5.0,2.0,0.5,...,0.450000,0.25,0.75,0.25,0.5,0.5,0.5,0.0,0.64,0.36


## Step 7 — Confirm that Python and R retained exactly the same ESS4 respondents

The comparison uses `(cntry, idno)` as the stable respondent key. Row order alone is not used as evidence of agreement.

Both the Python and R datasets must contain a unique key for every row, and their respondent-key sets must match exactly.

In [7]:
KEY_COLUMNS = ["cntry", "idno"]

ess4_python_for_comparison = ess4_without.copy()
ess4_python_for_comparison["idno"] = pd.to_numeric(
    ess4_python_for_comparison["idno"],
    errors="coerce",
).astype("Int64")
ess4_python_for_comparison["cntry"] = (
    ess4_python_for_comparison["cntry"].astype("string").str.strip()
)

assert ess4_python_for_comparison[KEY_COLUMNS].duplicated().sum() == 0

respondent_key_check = (
    ess4_python_for_comparison[KEY_COLUMNS]
    .merge(
        ess4_reference[KEY_COLUMNS],
        on=KEY_COLUMNS,
        how="outer",
        indicator=True,
        validate="one_to_one",
    )
)

respondent_key_counts = (
    respondent_key_check["_merge"]
    .value_counts()
    .rename_axis("merge_status")
    .rename("respondents")
    .to_frame()
)

display(respondent_key_counts)

python_only = respondent_key_check.loc[
    respondent_key_check["_merge"] == "left_only",
    KEY_COLUMNS,
]
reference_only = respondent_key_check.loc[
    respondent_key_check["_merge"] == "right_only",
    KEY_COLUMNS,
]

if not python_only.empty or not reference_only.empty:
    print("First Python-only respondent keys:")
    display(python_only.head(20))
    print("First R-reference-only respondent keys:")
    display(reference_only.head(20))
    raise AssertionError(
        "The Python and R ESS4 datasets do not contain the same respondent set."
    )

print(
    f"Respondent-set validation passed: all {len(ess4_without):,} "
    "ESS4 respondents match."
)

,respondents
merge_status,
both,45268
left_only,0
right_only,0


Respondent-set validation passed: all 45,268 ESS4 respondents match.


## Step 8 — Compare the harmonized ESS4 demographics

This is an additional diagnostic check. It confirms that the principal demographic fields retained by the Python curation agree with the corresponding fields in Van Noord's R object after factor-label harmonization.

In [8]:
comparison = ess4_python_for_comparison.merge(
    ess4_reference,
    on=KEY_COLUMNS,
    how="inner",
    suffixes=("_python", "_reference"),
    validate="one_to_one",
)

DEMOGRAPHIC_COLUMNS_TO_COMPARE = (
    "education_3cat",
    "hinctnta",
    "gndr",
    "agea",
    "rlgblg",
    "urbanization",
    "blgetmg",
)


def numeric_match_mask(
    python_values: pd.Series,
    reference_values: pd.Series,
    *,
    atol: float = 1e-12,
) -> pd.Series:
    python_numeric = pd.to_numeric(python_values, errors="coerce")
    reference_numeric = pd.to_numeric(reference_values, errors="coerce")

    both_missing = python_numeric.isna() & reference_numeric.isna()
    both_present = python_numeric.notna() & reference_numeric.notna()
    close = pd.Series(
        np.isclose(
            python_numeric.fillna(0.0),
            reference_numeric.fillna(0.0),
            rtol=0.0,
            atol=atol,
        ),
        index=python_numeric.index,
    )
    return both_missing | (both_present & close)


demographic_validation_rows = []
for column in DEMOGRAPHIC_COLUMNS_TO_COMPARE:
    python_column = f"{column}_python"
    reference_column = f"{column}_reference"
    match_mask = numeric_match_mask(
        comparison[python_column],
        comparison[reference_column],
    )
    demographic_validation_rows.append(
        {
            "variable": column,
            "respondents_compared": len(comparison),
            "mismatches": int((~match_mask).sum()),
            "all_values_match": bool(match_mask.all()),
        }
    )

demographic_validation = pd.DataFrame(demographic_validation_rows)
display(demographic_validation)

assert demographic_validation["all_values_match"].all()
print("All harmonized ESS4 demographic variables match the R reference.")

,variable,respondents_compared,mismatches,all_values_match
0,education_3cat,45268,0,True
1,hinctnta,45268,0,True
2,gndr,45268,0,True
3,agea,45268,0,True
4,rlgblg,45268,0,True
5,urbanization,45268,0,True
6,blgetmg,45268,0,True


All harmonized ESS4 demographic variables match the R reference.


## Step 9 — Compare all 19 ESS4 belief variables with the R reference

For every belief, the notebook compares:

- non-missing count;
- mean;
- sample standard deviation;
- respondent-level missingness pattern;
- respondent-level numeric value, with a strict absolute tolerance of `1e-12`.

The resulting validation table is saved as:

```text
data/processed/ess4_belief_validation_against_reference.csv
```

In [9]:
ESS4_NUMERIC_TOLERANCE = 1e-12

ess4_validation_rows = []

for belief in ess4.BELIEF_COLUMNS:
    python_values = pd.to_numeric(
        comparison[f"{belief}_python"],
        errors="coerce",
    )
    reference_values = pd.to_numeric(
        comparison[f"{belief}_reference"],
        errors="coerce",
    )

    match_mask = numeric_match_mask(
        python_values,
        reference_values,
        atol=ESS4_NUMERIC_TOLERANCE,
    )

    both_present = python_values.notna() & reference_values.notna()
    absolute_differences = (
        python_values.loc[both_present]
        - reference_values.loc[both_present]
    ).abs()

    maximum_absolute_difference = (
        float(absolute_differences.max())
        if not absolute_differences.empty
        else 0.0
    )

    ess4_validation_rows.append(
        {
            "belief_variable": belief,
            "N_python": int(python_values.notna().sum()),
            "N_reference": int(reference_values.notna().sum()),
            "mean_python": python_values.mean(),
            "mean_reference": reference_values.mean(),
            "sd_python": python_values.std(ddof=1),
            "sd_reference": reference_values.std(ddof=1),
            "missingness_mismatches": int(
                (python_values.isna() != reference_values.isna()).sum()
            ),
            "value_mismatches": int((~match_mask).sum()),
            "max_abs_difference": maximum_absolute_difference,
            "N_matches": bool(
                python_values.notna().sum() == reference_values.notna().sum()
            ),
            "values_match_within_tolerance": bool(match_mask.all()),
        }
    )

ess4_validation = pd.DataFrame(ess4_validation_rows)

assert ess4_validation["N_matches"].all()
assert ess4_validation["values_match_within_tolerance"].all()
assert (ess4_validation["missingness_mismatches"] == 0).all()
assert (
    ess4_validation["max_abs_difference"] <= ESS4_NUMERIC_TOLERANCE
).all()

ess4_validation.to_csv(ESS4_VALIDATION_PATH, index=False)

print("All 19 ESS4 belief variables match df_ESS4.RData respondent by respondent.")
print("Validation file written to:", ESS4_VALIDATION_PATH)
display(ess4_validation)

All 19 ESS4 belief variables match df_ESS4.RData respondent by respondent.
Validation file written to: /Users/karan/Desktop/SSM-MERC/polarization/data_curation/data/processed/ess4_belief_validation_against_reference.csv


,belief_variable,N_python,N_reference,mean_python,mean_reference,sd_python,sd_reference,missingness_mismatches,value_mismatches,max_abs_difference,N_matches,values_match_within_tolerance
0,left_right_identification,41465,41465,0.518816,0.518816,0.225549,0.225549,0,0,0.000000e+00,True,True
1,gender_inequality,45010,45010,0.566380,0.566380,0.257668,0.257668,0,0,0.000000e+00,True,True
2,anti_lgbt,44516,44516,0.335346,0.335346,0.310386,0.310386,0,0,0.000000e+00,True,True
3,euroscepticism,43342,43342,0.466132,0.466132,0.263316,0.263316,0,0,0.000000e+00,True,True
4,anti_immigration,44297,44297,0.481713,0.481713,0.269713,0.269713,0,0,1.110223e-16,True,True
5,anti_egalitarianism,44984,44984,0.307890,0.307890,0.212086,0.212086,0,0,0.000000e+00,True,True
6,benefits_harm_economy,43533,43533,0.515526,0.515526,0.225427,0.225427,0,0,0.000000e+00,True,True
7,benefits_harm_society,44693,44693,0.419406,0.419406,0.223687,0.223687,0,0,0.000000e+00,True,True
8,welfare_chauvinism,44309,44309,0.563802,0.563802,0.252744,0.252744,0,0,0.000000e+00,True,True
9,anti_economic_interventionism,44656,44656,0.229808,0.229808,0.158325,0.158325,0,0,2.220446e-16,True,True


## Step 10 — Compare the belief concepts shared by ESS4 and ESS8

ESS4 contains 19 constructed beliefs and ESS8 contains 20, but the item modules are not identical. Therefore, the cross-round comparison is restricted to the intersection of the two curated belief lists.

This is a descriptive comparison only. Differences in means, standard deviations, and missingness may reflect changes over time, different constituent items, different country coverage, or different respondents. They are not treated as validation failures.

In [10]:
SHARED_BELIEFS = tuple(
    belief
    for belief in ess8.BELIEF_COLUMNS
    if belief in ess4.BELIEF_COLUMNS
)

assert len(SHARED_BELIEFS) == 12

print("Shared belief concepts:", len(SHARED_BELIEFS))
for belief in SHARED_BELIEFS:
    print("-", belief)


def shared_belief_descriptives(
    dataframe: pd.DataFrame,
    beliefs: tuple[str, ...],
    *,
    round_label: str,
) -> pd.DataFrame:
    result = summarise_beliefs(dataframe, beliefs).rename(
        columns={
            "N_reproduced": "N_nonmissing",
            "mean_reproduced": "mean",
            "sd_reproduced": "sd",
        }
    )
    result.insert(0, "round", round_label)
    result["N_total"] = len(dataframe)
    result["N_missing"] = result["N_total"] - result["N_nonmissing"]
    result["percent_missing"] = (
        100.0 * result["N_missing"] / result["N_total"]
    )
    return result


cross_round_descriptives = pd.concat(
    [
        shared_belief_descriptives(
            ess4_without,
            SHARED_BELIEFS,
            round_label=ess4.ROUND_LABEL,
        ),
        shared_belief_descriptives(
            ess8_without,
            SHARED_BELIEFS,
            round_label=ess8.ROUND_LABEL,
        ),
    ],
    ignore_index=True,
)

cross_round_descriptives.to_csv(
    CROSS_ROUND_DESCRIPTIVES_PATH,
    index=False,
)

print("Cross-round descriptive table written to:", CROSS_ROUND_DESCRIPTIVES_PATH)
display(cross_round_descriptives)

Shared belief concepts: 12
- left_right_identification
- gender_inequality
- anti_lgbt
- euroscepticism
- anti_immigration
- anti_egalitarianism
- benefits_harm_economy
- benefits_harm_society
- welfare_chauvinism
- anti_economic_interventionism
- authoritarianism
- anti_libertarianism
Cross-round descriptive table written to: /Users/karan/Desktop/SSM-MERC/polarization/data_curation/data/processed/ess4_ess8_shared_belief_descriptives.csv


,round,belief_variable,N_nonmissing,mean,sd,N_total,N_missing,percent_missing
0,ESS Round 4,left_right_identification,41465,0.518816,0.225549,45268,3803,8.401078
1,ESS Round 4,gender_inequality,45010,0.566380,0.257668,45268,258,0.569939
2,ESS Round 4,anti_lgbt,44516,0.335346,0.310386,45268,752,1.661218
3,ESS Round 4,euroscepticism,43342,0.466132,0.263316,45268,1926,4.254661
4,ESS Round 4,anti_immigration,44297,0.481713,0.269713,45268,971,2.145003
5,ESS Round 4,anti_egalitarianism,44984,0.307890,0.212086,45268,284,0.627375
6,ESS Round 4,benefits_harm_economy,43533,0.515526,0.225427,45268,1735,3.832730
7,ESS Round 4,benefits_harm_society,44693,0.419406,0.223687,45268,575,1.270213
8,ESS Round 4,welfare_chauvinism,44309,0.563802,0.252744,45268,959,2.118494
9,ESS Round 4,anti_economic_interventionism,44656,0.229808,0.158325,45268,612,1.351948


## Step 11 — Inspect country coverage across the two rounds

Round 4 and Round 8 contain different country sets. This table reports the curated sample size in each round and should be used when interpreting cross-round descriptive differences.

In [11]:
ess4_country_counts = (
    ess4_without.groupby(["cntry", "country_name"], dropna=False)
    .size()
    .rename("N_ESS4")
    .reset_index()
)

ess8_country_counts = (
    ess8_without.groupby(["cntry", "country_name"], dropna=False)
    .size()
    .rename("N_ESS8")
    .reset_index()
)

country_coverage = ess4_country_counts.merge(
    ess8_country_counts,
    on=["cntry", "country_name"],
    how="outer",
    validate="one_to_one",
).sort_values("cntry", kind="stable")

country_coverage["present_in_ESS4"] = country_coverage["N_ESS4"].notna()
country_coverage["present_in_ESS8"] = country_coverage["N_ESS8"].notna()

display(country_coverage.reset_index(drop=True))

,cntry,country_name,N_ESS4,N_ESS8,present_in_ESS4,present_in_ESS8
0,AT,Austria,NaN,1766.0,False,True
1,BE,Belgium,1642.0,1672.0,True,True
2,BG,Bulgaria,1352.0,NaN,True,False
3,CH,Switzerland,1597.0,1318.0,True,True
4,CY,Cyprus,997.0,NaN,True,False
5,CZ,Czechia,1678.0,1920.0,True,True
6,DE,Germany,2483.0,2619.0,True,True
7,DK,Denmark,1461.0,NaN,True,False
8,EE,Estonia,1339.0,1836.0,True,True
9,ES,Spain,2031.0,1495.0,True,True


## Step 12 — Final validation summary

A successful run means that:

- all four curated files have the expected dimensions and schemas;
- the official ESS weights are complete and do not alter respondent-level belief data;
- all ESS8 published descriptive targets are reproduced;
- all ESS4 respondents and all 19 ESS4 belief values agree with `df_ESS4.RData`;
- the shared-belief cross-round descriptive table has been written successfully.

In [12]:
final_summary = pd.DataFrame(
    [
        {
            "check": "ESS4 weighted/unweighted internal validation",
            "status": "passed",
        },
        {
            "check": "ESS8 weighted/unweighted internal validation",
            "status": "passed",
        },
        {
            "check": "ESS8 Supplementary Table A2 validation",
            "status": "passed",
        },
        {
            "check": "ESS4 respondent set against df_ESS4.RData",
            "status": "passed",
        },
        {
            "check": "ESS4 demographics against df_ESS4.RData",
            "status": "passed",
        },
        {
            "check": "ESS4 beliefs against df_ESS4.RData",
            "status": "passed",
        },
        {
            "check": "ESS4/ESS8 shared-belief descriptive export",
            "status": "passed",
        },
    ]
)

display(final_summary)

print("Validation completed successfully.")
print()
print("Generated validation files:")
print("-", ESS4_VALIDATION_PATH)
print("-", ESS8_VALIDATION_PATH)
print("-", CROSS_ROUND_DESCRIPTIVES_PATH)

,check,status
0,ESS4 weighted/unweighted internal validation,passed
1,ESS8 weighted/unweighted internal validation,passed
2,ESS8 Supplementary Table A2 validation,passed
3,ESS4 respondent set against df_ESS4.RData,passed
4,ESS4 demographics against df_ESS4.RData,passed
5,ESS4 beliefs against df_ESS4.RData,passed
6,ESS4/ESS8 shared-belief descriptive export,passed


Validation completed successfully.

Generated validation files:
- /Users/karan/Desktop/SSM-MERC/polarization/data_curation/data/processed/ess4_belief_validation_against_reference.csv
- /Users/karan/Desktop/SSM-MERC/polarization/data_curation/data/processed/ess8_belief_validation_against_paper.csv
- /Users/karan/Desktop/SSM-MERC/polarization/data_curation/data/processed/ess4_ess8_shared_belief_descriptives.csv
